# IRDAS — Phase 2: DR Pseudo-Label Generation
> Load the Phase 1 DR Teacher (QWK 0.9342) → Run D4 TTA on HRDC images → Save `hrdc_pseudo_labels.csv` with DR grade + confidence for every HRDC image.

In [1]:
# ============================================================
# ⚙️  CELL 1 — Environment Lock
# ============================================================
import os
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [2]:
# ============================================================
# 📦  CELL 2 — Imports
# ============================================================
import gc
import ctypes
import warnings
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

warnings.filterwarnings('ignore')
cv2.setNumThreads(0)
torch.backends.cudnn.benchmark = True

In [3]:
# ============================================================
# ⚙️  CELL 3 — Paths & Config
# ============================================================
# ─── EDIT THESE to match your Kaggle dataset input paths ───
CFG = {
    # ── Phase 1 artifacts ──────────────────────────────────
    # After uploading dr_teacher_weights.zip as a Kaggle dataset,
    # the path will be something like:
    #   /kaggle/input/<your-dataset-name>/checkpoints/best_ema_teacher.pth
    # Adjust to match your actual dataset slug.
    # 'teacher_ckpt':  '/kaggle/input/dr-teacher-weights/checkpoints/best_ema_teacher.pth',
    # 'thresholds':    '/kaggle/input/dr-teacher-weights/checkpoints/optimal_thresholds.npy',

    # # ── HRDC dataset ───────────────────────────────────────
    # # Add the HRDC Kaggle dataset:
    # #   https://www.kaggle.com/datasets/gunavenkatdoddi/eye-diseases-classification
    # # OR the official HRDC challenge dataset if you have it.
    # # The script auto-detects image folder structure (see HRDCDataset below).
    # 'hrdc_root':     '/kaggle/input/hrdc-hypertensive-retinopathy-grading-challenge',
    # 'hrdc_csv':      None,   # Set to CSV path if it exists, else None (auto-scan)


    'teacher_ckpt': '/kaggle/input/datasets/nawazishbilal/dr-teacher-weights/ema_final.pth',

    'thresholds': '/kaggle/input/datasets/nawazishbilal/dr-teacher-weights/best_thresholds.npy',

    # ── HRDC dataset ───────────────────────────────────────
    'hrdc_root': '/kaggle/input/datasets/nawazishbilal/hrdc-hypertensive-retinopathy-grading-challenge',

    'hrdc_csv': None,

    
    # ── Inference config ───────────────────────────────────
    'img_size':      512,    # Match Phase 1 stage-3 resolution
    'batch_size':    8,      # Reduce to 4 if OOM on T4
    'num_workers':   2,
    'device':        'cuda' if torch.cuda.is_available() else 'cpu',

    # ── Architecture (MUST match Phase 1 exactly) ──────────
    'backbone':        'tf_efficientnet_b4.ns_jft_in1k',
    'drop_path_rate':  0.2,
    'bifpn_channels':  256,
    'bifpn_layers':    2,
    'msd_k':           5,
    'dropout':         0.3,

    # ── Confidence filtering ───────────────────────────────
    # Images with confidence below this are flagged (not removed).
    # You can filter them out before Phase 3 if desired.
    'confidence_threshold': 0.70,

    # ── Output ─────────────────────────────────────────────
    'out_dir':  '/kaggle/working',
    'out_csv':  '/kaggle/working/hrdc_pseudo_labels.csv',
}

NUM_CLASSES  = 5
CORAL_LEVELS = 4   # = NUM_CLASSES - 1

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

os.makedirs(CFG['out_dir'], exist_ok=True)
print(f"Device: {CFG['device']}")
if CFG['device'] == 'cuda':
    print(f"GPU   : {torch.cuda.get_device_name(0)}")


# ============================================================
# 🏗️  CELL 4 — Model Architecture
#    ⚠️  MUST be byte-for-byte identical to Phase 1 definitions.
#    If you see "unexpected key" or "missing key" errors, compare
#    these classes with the Phase 1 notebook and fix accordingly.
# ============================================================

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        mid = max(in_planes // ratio, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, mid, 1, bias=False), nn.ReLU(),
            nn.Conv2d(mid, in_planes, 1, bias=False))

    def forward(self, x):
        return x * torch.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))


class SpatialAttention(nn.Module):
    def __init__(self, ks=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, ks, padding=(ks-1)//2, bias=False)

    def forward(self, x):
        avg = torch.mean(x, 1, keepdim=True)
        mx, _ = torch.max(x, 1, keepdim=True)
        return x * torch.sigmoid(self.conv(torch.cat([avg, mx], 1)))


class CBAM(nn.Module):
    def __init__(self, p):
        super().__init__()
        self.ca = ChannelAttention(p)
        self.sa = SpatialAttention()

    def forward(self, x):
        return self.sa(self.ca(x))


class DWConv(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.dw = nn.Conv2d(ch, ch, 3, padding=1, groups=ch, bias=False)
        self.pw = nn.Conv2d(ch, ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(ch)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.bn(self.pw(self.dw(x))))


class BiFPNLayer(nn.Module):
    def __init__(self, ch=256, eps=1e-4):
        super().__init__()
        self.eps = eps
        self.w_p4_td  = nn.Parameter(torch.ones(2))
        self.w_p3_out = nn.Parameter(torch.ones(2))
        self.w_p4_out = nn.Parameter(torch.ones(3))
        self.w_p5_out = nn.Parameter(torch.ones(2))
        self.conv_p4_td  = DWConv(ch)
        self.conv_p3_out = DWConv(ch)
        self.conv_p4_out = DWConv(ch)
        self.conv_p5_out = DWConv(ch)

    def _up(self, x, t):
        return F.interpolate(x, t.shape[-2:], mode='nearest')

    def _dn(self, x, t):
        return F.adaptive_avg_pool2d(x, t.shape[-2:])

    def forward(self, p3, p4, p5):
        w4  = F.relu(self.w_p4_td.clone());  w4  = w4  / (w4.sum()  + self.eps)
        w3  = F.relu(self.w_p3_out.clone()); w3  = w3  / (w3.sum()  + self.eps)
        w4o = F.relu(self.w_p4_out.clone()); w4o = w4o / (w4o.sum() + self.eps)
        w5o = F.relu(self.w_p5_out.clone()); w5o = w5o / (w5o.sum() + self.eps)
        p4_td  = self.conv_p4_td(w4[0]*p4  + w4[1]*self._up(p5, p4))
        p3_out = self.conv_p3_out(w3[0]*p3 + w3[1]*self._up(p4_td, p3))
        p4_out = self.conv_p4_out(w4o[0]*p4 + w4o[1]*p4_td + w4o[2]*self._dn(p3_out, p4))
        p5_out = self.conv_p5_out(w5o[0]*p5 + w5o[1]*self._dn(p4_out, p5))
        return p3_out, p4_out, p5_out


class BiFPN(nn.Module):
    def __init__(self, in_ch, out_ch=256, n=2):
        super().__init__()
        self.lat = nn.ModuleList([
            nn.Sequential(nn.Conv2d(c, out_ch, 1, bias=False),
                          nn.BatchNorm2d(out_ch), nn.SiLU())
            for c in in_ch
        ])
        self.layers = nn.ModuleList([BiFPNLayer(out_ch) for _ in range(n)])

    def forward(self, p3r, p4r, p5r):
        p3, p4, p5 = self.lat[0](p3r), self.lat[1](p4r), self.lat[2](p5r)
        for layer in self.layers:
            p3, p4, p5 = layer(p3, p4, p5)
        return p3, p4, p5


class GeMPooling(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(self.eps).pow(self.p), x.shape[-2:]
        ).pow(1.0 / self.p)


class DRTeacher(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg['backbone'], pretrained=False,
            features_only=True, out_indices=(2, 3, 4),
            drop_path_rate=cfg['drop_path_rate'])

        ch = self.backbone.feature_info.channels()
        oc = cfg['bifpn_channels']
        self.bifpn    = BiFPN(ch, oc, cfg['bifpn_layers'])
        self.pool     = GeMPooling()
        self.cbam_p3  = CBAM(oc)
        self.cbam_p5  = CBAM(oc)
        self.dropout  = nn.Dropout(cfg['dropout'])
        self.head     = nn.Linear(oc * 2, CORAL_LEVELS)
        self.msd_k    = cfg['msd_k']

    def forward(self, x):
        f = self.backbone(x)
        p3, p4, p5 = self.bifpn(f[0], f[1], f[2])
        feat = torch.cat([
            self.pool(self.cbam_p3(p3)).flatten(1),
            self.pool(self.cbam_p5(p5)).flatten(1)
        ], 1)
        if self.training:
            return torch.stack([
                self.head(self.dropout(feat)) for _ in range(self.msd_k)
            ]).mean(0)
        return self.head(feat)
# ── CORAL helpers ───────────────────────────────────────────
def coral_logits_to_probs(logits: torch.Tensor) -> torch.Tensor:
    """Sigmoid each CORAL binary output → cumulative P(Y > k)."""
    return torch.sigmoid(logits)          # (B, 4)

def coral_probs_to_score(probs: torch.Tensor) -> torch.Tensor:
    """Sum P(Y > k) across k → continuous ordinal score in [0, 4]."""
    return probs.sum(dim=1)               # (B,)

def score_to_grade(scores: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    """Apply optimal thresholds [t0, t1, t2, t3] to assign grade 0-4."""
    grades = np.zeros(len(scores), dtype=int)
    for i, t in enumerate(thresholds):
        grades[scores > t] = i + 1
    return grades

def grade_confidence(scores: np.ndarray, grades: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    """
    Confidence = normalised distance from the nearest threshold boundary.
    Score exactly at a threshold → confidence 0.
    Score far from all thresholds → confidence 1.
    """
    # Build extended thresholds: -inf, t0, t1, t2, t3, +inf
    t_ext = np.array([-np.inf] + list(thresholds) + [np.inf])
    confs = []
    for s, g in zip(scores, grades):
        lo = t_ext[g]          # lower boundary of predicted grade bin
        hi = t_ext[g + 1]      # upper boundary
        bin_width = hi - lo if np.isfinite(hi - lo) else 4.0
        # Distance from nearest edge, normalised by half-bin-width
        dist_lo = s - lo if np.isfinite(lo) else bin_width
        dist_hi = hi - s if np.isfinite(hi) else bin_width
        margin = min(dist_lo, dist_hi)
        conf = float(np.clip(margin / (0.5 * bin_width + 1e-6), 0.0, 1.0))
        confs.append(conf)
    return np.array(confs)

Device: cuda
GPU   : Tesla T4


In [4]:
# ============================================================
# 🔍  CELL 5 — Load Checkpoint & Verify
# ============================================================
def load_teacher(cfg: dict) -> tuple:
    """
    Load EMA teacher weights into DRTeacher.
    Handles three possible save formats:
      A) state_dict directly
      B) {'model': state_dict, 'thresholds': ...}
      C) AveragedModel wrapper (keys prefixed with 'module.')
    Returns (model, thresholds).
    """
    device = cfg['device']
    model  = DRTeacher(cfg).to(device)
    model.eval()

    ckpt_path = cfg['teacher_ckpt']
    thr_path  = cfg['thresholds']

    print(f"\n📂 Loading checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device)

    # ── Determine state dict ────────────────────────────────
    if isinstance(ckpt, dict) and 'model' in ckpt:
        state_dict = ckpt['model']
        print("   Format detected: {'model': sd, ...}")
    elif isinstance(ckpt, dict) and any(k.startswith('module.') for k in ckpt):
        # AveragedModel wrapper
        state_dict = {k.replace('module.', '', 1): v for k, v in ckpt.items()}
        print("   Format detected: AveragedModel (module. prefix stripped)")
    else:
        state_dict = ckpt
        print("   Format detected: raw state_dict")

    # ── Key compatibility check ─────────────────────────────
    model_keys  = set(model.state_dict().keys())
    ckpt_keys   = set(state_dict.keys())
    missing     = model_keys - ckpt_keys
    unexpected  = ckpt_keys - model_keys

    if missing or unexpected:
        print(f"\n⚠️  KEY MISMATCH DETECTED")
        if missing:
            print(f"   Missing  ({len(missing)}): {sorted(missing)[:10]}")
        if unexpected:
            print(f"   Unexpected ({len(unexpected)}): {sorted(unexpected)[:10]}")
        print("\n   💡 FIX: The architecture above doesn't exactly match Phase 1.")
        print("   Open the Phase 1 notebook, copy-paste the BiFPN / MSDPooling")
        print("   / CoralHead classes into Cell 4 of this notebook, then re-run.")
    else:
        print(f"   ✅ All {len(model_keys)} keys matched perfectly.")

    model.load_state_dict(state_dict, strict=False)

    # ── Load thresholds ─────────────────────────────────────
    print(f"\n📂 Loading thresholds: {thr_path}")
    thresholds = np.load(thr_path)
    print(f"   Thresholds: {thresholds}")

    return model, thresholds

In [5]:
# ============================================================
# 📁  CELL 6 — HRDC Dataset
# ============================================================

# HRDC image folder search order (most common Kaggle structures)
_HRDC_IMG_CANDIDATES = [
    "Training_Dataset",
    "training/Training_Dataset",
    "train",
    "Train",
    "images",
    "Images",
    "train_images",
]

def _find_hrdc_images(root: str) -> list:
    """
    Auto-discover all JPEG/PNG images under the HRDC root.
    Returns sorted list of absolute paths.
    """
    root = Path(root)
    imgs = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
        imgs.extend(root.rglob(ext))
    imgs = sorted(set(imgs))
    print(f"   Found {len(imgs)} images under {root}")
    return imgs


def _find_hrdc_csv(root: str) -> pd.DataFrame | None:
    """Try common CSV locations inside the HRDC dataset."""
    root = Path(root)
    for pattern in ('*.csv', '**/*.csv'):
        csvs = sorted(root.glob(pattern))
        for p in csvs:
            try:
                df = pd.read_csv(p)
                print(f"   Found CSV: {p}  shape={df.shape}  cols={list(df.columns)}")
                return df
            except Exception:
                pass
    return None


class HRDCDataset(Dataset):
    """
    Inference-only dataset for HRDC images.
    Returns (tensor, image_id_str) pairs.
    """

    def __init__(self, image_paths: list, img_size: int):
        self.paths    = image_paths
        self.img_size = img_size
        self.tf = A.Compose([
            A.LongestMaxSize(max_size=img_size),
            A.PadIfNeeded(img_size, img_size,
                          border_mode=cv2.BORDER_CONSTANT, value=0),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img  = cv2.imread(str(path))
        if img is None:
            # Return black image if read fails
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = self.tf(image=img)['image']
        return tensor, Path(path).stem   # (C, H, W), image_id

In [6]:
# ============================================================
# 🔄  CELL 7 — D4 TTA (8 augmentations)
# ============================================================

def _d4_augment(img: torch.Tensor, aug_id: int) -> torch.Tensor:
    """
    Apply one of 8 D4 group transforms.
    aug_id 0 = identity, 1-3 = rotations, 4-7 = flips + rotations.
    """
    k     = aug_id % 4
    flip  = aug_id >= 4
    if flip:
        img = torch.flip(img, dims=[-1])   # horizontal flip
    if k > 0:
        img = torch.rot90(img, k=k, dims=[-2, -1])
    return img


@torch.no_grad()
def predict_with_tta(model: nn.Module,
                     loader: DataLoader,
                     device: str,
                     n_tta: int = 8) -> dict:
    """
    Run inference with D4 TTA.
    Returns dict: image_id → {'coral_score': float, 'probs': np.array(4)}
    """
    model.eval()
    results = {}

    for batch_imgs, batch_ids in tqdm(loader, desc="Inference"):
        batch_imgs = batch_imgs.to(device)           # (B, C, H, W)
        B = batch_imgs.shape[0]

        # Accumulate logit sums across TTA views
        logit_acc = torch.zeros(B, CORAL_LEVELS, device=device)

        for aug_id in range(n_tta):
            aug_batch = torch.stack([_d4_augment(batch_imgs[i], aug_id)
                                     for i in range(B)])
            logits = model(aug_batch)                # (B, 4)
            logit_acc += logits

        # Average logits (better than averaging probs for CORAL)
        avg_logits = logit_acc / n_tta               # (B, 4)
        probs      = coral_logits_to_probs(avg_logits)   # (B, 4)
        scores     = coral_probs_to_score(probs)         # (B,)

        for i, img_id in enumerate(batch_ids):
            results[img_id] = {
                'coral_score': float(scores[i].cpu()),
                'probs': probs[i].cpu().numpy(),
            }

    return results

In [7]:
# ============================================================
# 🏷️  CELL 8 — Generate Pseudo-Labels
# ============================================================

def generate_pseudo_labels(cfg: dict) -> pd.DataFrame:
    """Full pipeline: load → discover images → infer → threshold → save."""

    # ── Load model + thresholds ─────────────────────────────
    model, thresholds = load_teacher(cfg)

    # ── Discover HRDC images ────────────────────────────────
    print(f"\n📂 Scanning HRDC root: {cfg['hrdc_root']}")
    img_paths = _find_hrdc_images(cfg['hrdc_root'])

    # Try to find ground-truth HR labels (for reference only, not used in inference)
    hr_df = None
    if cfg.get('hrdc_csv'):
        hr_df = pd.read_csv(cfg['hrdc_csv'])
        print(f"   GT CSV loaded: {len(hr_df)} rows")
    else:
        hr_df = _find_hrdc_csv(cfg['hrdc_root'])

    if len(img_paths) == 0:
        raise FileNotFoundError(
            f"No images found under {cfg['hrdc_root']}.\n"
            "Check that the HRDC dataset is correctly added to this Kaggle notebook\n"
            "under 'Add Data' → your HRDC dataset."
        )

    # ── Build DataLoader ─────────────────────────────────────
    dataset = HRDCDataset(img_paths, cfg['img_size'])
    loader  = DataLoader(
        dataset,
        batch_size  = cfg['batch_size'],
        num_workers = cfg['num_workers'],
        pin_memory  = True,
        shuffle     = False,
    )

    print(f"\n🔄 Running D4 TTA inference on {len(dataset)} images...")
    results = predict_with_tta(model, loader, cfg['device'])

    # ── Apply thresholds ─────────────────────────────────────
    image_ids     = list(results.keys())
    coral_scores  = np.array([results[i]['coral_score'] for i in image_ids])
    all_probs     = np.stack([results[i]['probs'] for i in image_ids])  # (N, 4)

    pseudo_grades = score_to_grade(coral_scores, thresholds)
    confidences   = grade_confidence(coral_scores, pseudo_grades, thresholds)

    # ── Build DataFrame ──────────────────────────────────────
    df = pd.DataFrame({
        'image_id':        image_ids,
        'pseudo_dr_grade': pseudo_grades,
        'confidence':      np.round(confidences, 4),
        'coral_score':     np.round(coral_scores, 4),
        'p0':              np.round(all_probs[:, 0], 4),   # P(Y > 0)
        'p1':              np.round(all_probs[:, 1], 4),   # P(Y > 1)
        'p2':              np.round(all_probs[:, 2], 4),   # P(Y > 2)
        'p3':              np.round(all_probs[:, 3], 4),   # P(Y > 3)
        'low_confidence':  confidences < cfg['confidence_threshold'],
    })

    # ── Optionally merge HR ground-truth labels ───────────────
    if hr_df is not None:
        # Try to find the right column names automatically
        id_col = None
        hr_col = None
        for c in hr_df.columns:
            if 'id' in c.lower() or 'name' in c.lower() or 'file' in c.lower():
                id_col = c
            if 'hr' in c.lower() or 'hyper' in c.lower() or 'grade' in c.lower() or 'label' in c.lower():
                hr_col = c
        if id_col and hr_col:
            hr_df = hr_df[[id_col, hr_col]].rename(
                columns={id_col: 'image_id', hr_col: 'hr_grade'}
            )
            hr_df['image_id'] = hr_df['image_id'].astype(str).str.replace(r'\.(jpg|jpeg|png)', '', regex=True)
            df = df.merge(hr_df, on='image_id', how='left')
            print(f"\n   Merged HR ground-truth labels (col '{hr_col}').")
        else:
            print(f"\n   ⚠️  Could not auto-detect id/hr columns. Columns: {list(hr_df.columns)}")
            print("   Manually set hr_df merge after saving if needed.")

    # ── Save ─────────────────────────────────────────────────
    df.to_csv(cfg['out_csv'], index=False)
    return df

In [ ]:
# ============================================================
# 📊  CELL 9 — Run & Summarise
# ============================================================
if __name__ == '__main__':
    df = generate_pseudo_labels(CFG)

    print("\n" + "=" * 60)
    print("  PSEUDO-LABEL GENERATION COMPLETE")
    print("=" * 60)
    print(f"  Total images labelled : {len(df)}")
    print(f"  Output saved to       : {CFG['out_csv']}")

    print("\n  DR Grade Distribution:")
    grade_counts = df['pseudo_dr_grade'].value_counts().sort_index()
    for g, n in grade_counts.items():
        bar = '█' * int(30 * n / len(df))
        pct = 100 * n / len(df)
        print(f"    Grade {g}: {n:5d} ({pct:5.1f}%)  {bar}")

    print(f"\n  Low-confidence images : {df['low_confidence'].sum()} "
          f"({100 * df['low_confidence'].mean():.1f}% below {CFG['confidence_threshold']})")
    print(f"  Mean confidence       : {df['confidence'].mean():.4f}")
    print(f"  Mean CORAL score      : {df['coral_score'].mean():.4f}")

    if 'hr_grade' in df.columns:
        print(f"\n  HR Grade Distribution (ground-truth):")
        for g, n in df['hr_grade'].value_counts().sort_index().items():
            print(f"    HR {g}: {n}")

    print("\n  Sample output:")
    print(df.head(10).to_string(index=False))
    print("=" * 60)

    # ── Quick sanity checks ───────────────────────────────────
    assert df['pseudo_dr_grade'].between(0, 4).all(), "Grade out of range!"
    assert df['confidence'].between(0, 1).all(),      "Confidence out of range!"
    print("\n  ✅ Sanity checks passed.")
    print("\n  📋 NEXT STEP → Phase 3 (MSDNet multi-task training)")
    print("     hrdc_pseudo_labels.csv is ready to be merged as DR labels.")


📂 Loading checkpoint: /kaggle/input/datasets/nawazishbilal/dr-teacher-weights/ema_final.pth
   Format detected: raw state_dict

⚠️  KEY MISMATCH DETECTED
   Missing  (128): ['bifpn.laterals.0.0.weight', 'bifpn.laterals.0.1.bias', 'bifpn.laterals.0.1.num_batches_tracked', 'bifpn.laterals.0.1.running_mean', 'bifpn.laterals.0.1.running_var', 'bifpn.laterals.0.1.weight', 'bifpn.laterals.1.0.weight', 'bifpn.laterals.1.1.bias', 'bifpn.laterals.1.1.num_batches_tracked', 'bifpn.laterals.1.1.running_mean']
   Unexpected (90): ['bifpn.lat.0.0.weight', 'bifpn.lat.0.1.bias', 'bifpn.lat.0.1.num_batches_tracked', 'bifpn.lat.0.1.running_mean', 'bifpn.lat.0.1.running_var', 'bifpn.lat.0.1.weight', 'bifpn.lat.1.0.weight', 'bifpn.lat.1.1.bias', 'bifpn.lat.1.1.num_batches_tracked', 'bifpn.lat.1.1.running_mean']

   💡 FIX: The architecture above doesn't exactly match Phase 1.
   Open the Phase 1 notebook, copy-paste the BiFPN / MSDPooling
   / CoralHead classes into Cell 4 of this notebook, then re-run.



Inference:  89%|████████▉ | 79/89 [02:07<00:15,  1.57s/it]